# Create the small app with LangChain and OpenAI

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")
os.environ['LANGCHAIN_TRACING_V2'] = "true"

In [6]:
# Step 1: Data Ingestion - Scrap the data from the websites
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/administration/tutorials/manage_spend")
docs = loader.load()
docs


[Document(metadata={'source': 'https://docs.smith.langchain.com/administration/tutorials/manage_spend', 'title': 'Optimize tracing spend on LangSmith | 🦜️🛠️ LangSmith', 'description': 'Before diving into this content, it might be helpful to read the following:', 'language': 'en'}, page_content='\n\n\n\n\nOptimize tracing spend on LangSmith | 🦜️🛠️ LangSmith\n\n\n\n\n\n\n\n\nSkip to main contentOur Building Ambient Agents with LangGraph course is now available on LangChain Academy!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppGet StartedObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Platform)AdministrationTutorialsOptimize tracing spend on LangSmithHow-to GuidesSetupConceptual GuideSelf-hostingPricingReferenceCloud architecture and scalabilityAuthz and AuthnAuthentication methodsdata_formatsEvaluationDataset transformationsRegions FAQsdk_referenceAdministrationTutorialsOptimize tracing spend on LangSmithOn this pageOptimize tracing spend on LangSmith\nRecommended R

In [12]:
# Step 2: Divide the documents(text) into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

# Step 3: Convert the text(documents) into vector and store the vectors in "Vectore DB"
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embedding = OpenAIEmbeddings()
vector_store_db = FAISS.from_documents(documents, embedding)

# Step 4: Execute the query to get an result
query = "LangSmith has two usage limits: total traces and extended"
result = vector_store_db.similarity_search(query)
result[0].page_content



OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [10]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
# Retrieval Chain, Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Prompt
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
 {context}
</context>
"""
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain.invoke(
    {
        "input": "LangSmith has two usage limits: total traces and extended",
        "context": [Document(page_content="This tutorial walks through optimizing your spend on LangSmith. In it, we will learn how to optimize existing spend and prevent future overspend on a realistic real-world example. We will use an existing LangSmith organization with high usage. Concepts can be transferred to your own organization.")]
    }
)


However, we want the document to first come from the retriver we just set up. That way, we can use the retriever to dynamically select themost relevant documents and pass those in for given question.

In [ ]:
# Retriver: It is an interface.Pass the input to the retriver and get the response from the vectordb
# Input =---> Retriver -----> VectorStoredb
from langchain.chains import create_retrieval_chain
retriver = vector_store_db.as_retriever()
retriver_chain = create_retrieval_chain(retriver, document_chain)

In [ ]:
# Get the response from the LLM
response = retriver_chain.invoke({
    "input": "LangSmith has two usage limits: total traces and extended",
})
response["answer"]